# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dushyant1607/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

**Unit of analysis:** One row represents one content page for one reporting period, if the warehouse schema confirms that grain.

**Time window:** March 2026 (`2026-03`), a mid-panel month used to develop features and investigate the label. June 2026 remains reserved as a sealed test month.

**Decision:** Rank eligible content pages for human review to help prioritize potential refresh opportunities. The final refresh decision remains with the human reviewer.

In [17]:
%pip -q install duckdb huggingface_hub

from pathlib import Path
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

# 1. Get the Hugging Face token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN is missing. Add it in Colab Secrets "
        "and enable notebook access."
    )

# 2. Download only the March 2026 partition
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token,
)

print("March 2026 file downloaded:", Path(march_file).exists())

# 3. Connect to DuckDB and create a view
con = duckdb.connect()

# Escape quotes in the local file path
quoted_path = str(march_file).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW march AS
    SELECT *
    FROM read_parquet('{quoted_path}')
""")

# 4. Query 1: verify the row grain
grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            report_date,
            client_hash_id,
            content_hash_id
        )) AS distinct_grain_keys
    FROM march
""").df()

display(grain_check)

# 5. Interpret the result
total_rows = int(grain_check.loc[0, "total_rows"])
distinct_keys = int(grain_check.loc[0, "distinct_grain_keys"])

if total_rows == distinct_keys:
    print(
        "Verified: every row has a unique "
        "report_date × client × content combination."
    )
else:
    print(
        f"Warning: {total_rows - distinct_keys:,} rows "
        "are beyond the distinct grain-key count. "
        "Investigate duplicates before claiming the grain is unique."
    )

March 2026 file downloaded: True


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


Verified: every row has a unique report_date × client × content combination.


## 2. Fields: feature / label / context / excluded

**Features:** Candidate fields describing page performance and history that are available at the decision moment. I will select no more than five after inspecting the actual schema.

**Label:** A clearly defined decline or refresh-opportunity outcome, using the warehouse's documented label or a defensible proxy.

**Context:** Reporting month, anonymized page/client identifiers, and availability indicators needed to validate the data slice.

**Excluded:** Label-derived fields and any information recorded after the decision moment, because they can leak the answer into the model. Client-identifying details, URLs, and private queries will not be included in public outputs.

In [18]:
# SECTION 2 — Fields: feature / label / context / excluded

# 1. Inspect the available fields and their data types
schema = con.execute("""
    DESCRIBE SELECT * FROM march
""").df()

print("AVAILABLE WAREHOUSE FIELDS")
display(schema)

# 2. Summarize the data coverage for the selected March slice
coverage = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM march
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("DATA COVERAGE")
display(coverage)

# 3. Define five candidate features using GSC-available rows
# These are candidates, not yet validated predictors of a future outcome.
feature_frame = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        gsc_sum_position,
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks::DOUBLE / gsc_impressions
            ELSE NULL
        END AS gsc_ctr
    FROM march
    WHERE gsc_data_available IS TRUE
""").df()

print("FEATURE FRAME — FIRST 10 ROWS")
display(feature_frame.head(10))

print(f"Feature-frame rows: {len(feature_frame):,}")

# 4. Document the five features and their availability
feature_notes = [
    {
        "feature": "gsc_impressions",
        "available_when": (
            "When GSC measurements for the reporting date are available."
        ),
    },
    {
        "feature": "gsc_clicks",
        "available_when": (
            "When GSC measurements for the reporting date are available."
        ),
    },
    {
        "feature": "gsc_avg_position",
        "available_when": (
            "When the GSC position aggregate for the reporting date is available."
        ),
    },
    {
        "feature": "gsc_sum_position",
        "available_when": (
            "When the GSC position aggregate for the reporting date is available."
        ),
    },
    {
        "feature": "gsc_ctr",
        "available_when": (
            "Calculated from clicks and impressions for the same reporting date."
        ),
    },
]

import pandas as pd

feature_notes_df = pd.DataFrame(feature_notes)
print("FEATURE AVAILABILITY NOTES")
display(feature_notes_df)

print("""
CONTRACT CLASSIFICATION
Features: the five candidate fields listed above.
Label: not yet defined; requires a separate future outcome window.
Context: report_date, client_hash_id, content_hash_id.
Excluded: label-derived fields, future measurements, and client-identifying details.

Important: these are same-day candidate features. Before modeling, verify that
they would actually be available at the intended decision time.
""")

AVAILABLE WAREHOUSE FIELDS


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


DATA COVERAGE


,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FEATURE FRAME — FIRST 10 ROWS


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,gsc_ctr
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,67,0.000000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,616,0.008000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,28,0.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,25,0.000000
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,1756,0.004184
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,1496,0.000000
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,180,0.000000
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,434,0.000000
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,9,0.000000


Feature-frame rows: 3,611,061
FEATURE AVAILABILITY NOTES


,feature,available_when
0,gsc_impressions,When GSC measurements for the reporting date a...
1,gsc_clicks,When GSC measurements for the reporting date a...
2,gsc_avg_position,When the GSC position aggregate for the report...
3,gsc_sum_position,When the GSC position aggregate for the report...
4,gsc_ctr,Calculated from clicks and impressions for the...



CONTRACT CLASSIFICATION
Features: the five candidate fields listed above.
Label: not yet defined; requires a separate future outcome window.
Context: report_date, client_hash_id, content_hash_id.
Excluded: label-derived fields, future measurements, and client-identifying details.

Important: these are same-day candidate features. Before modeling, verify that
they would actually be available at the intended decision time.



In [19]:
# PROVISIONAL FUTURE-OUTCOME LABEL
# March features; April outcome. June remains sealed.

from huggingface_hub import hf_hub_download
from pathlib import Path

# Download April 2026 partition
april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    token=hf_token,
)

quoted_april = str(april_file).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW april AS
    SELECT *
    FROM read_parquet('{quoted_april}')
""")

# Aggregate monthly GSC measurements per client/content pair
monthly = con.execute("""
    WITH march_month AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            AVG(gsc_avg_position) AS march_avg_position,
            COUNT(*) AS march_days,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS march_available_days
        FROM march
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-04-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april_month AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS april_available_days
        FROM april
        WHERE report_date >= DATE '2026-04-01'
          AND report_date < DATE '2026-05-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.march_impressions,
        m.march_clicks,
        m.march_avg_position,
        m.march_days,
        m.march_available_days,
        a.april_impressions,
        a.april_available_days,
        CASE
            WHEN m.march_impressions > 0
             AND a.april_impressions < 0.8 * m.march_impressions
            THEN 1
            WHEN m.march_impressions > 0
            THEN 0
            ELSE NULL
        END AS future_decline_proxy
    FROM march_month m
    INNER JOIN april_month a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id
    WHERE m.march_available_days > 0
      AND a.april_available_days > 0
""").df()

display(monthly.head(10))
print("Page pairs with both months available:", f"{len(monthly):,}")
print(
    "Future-decline proxy rate:",
    f"{monthly['future_decline_proxy'].mean():.1%}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_days,march_available_days,april_impressions,april_available_days,future_decline_proxy
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,24,24,27.0,19,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,8.240351,31,31,16121.0,30,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,6.015432,27,27,81.0,22,0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,5.956862,31,31,213.0,29,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,12.977513,21,21,23.0,15,1
5,client_62f4a7e64f5e0096,content_df22bda1218f13ff,2099.0,1.0,3.066796,31,31,1472.0,30,1
6,client_62f4a7e64f5e0096,content_aa184c1b4ea518e5,288.0,0.0,9.444919,30,30,185.0,30,1
7,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,3535.0,25.0,2.951764,31,31,10500.0,30,0
8,client_62f4a7e64f5e0096,content_d7568011c4325a33,1558.0,3.0,5.264774,31,31,746.0,30,1
9,client_62f4a7e64f5e0096,content_e847a4dcc8af3742,2021.0,0.0,7.653508,31,31,675.0,30,1


Page pairs with both months available: 158,549
Future-decline proxy rate: 47.8%


## 3. Verify it with queries (grain, counts, missing values, windows)

**Data limitations**

This is daily page-level performance data, not a direct measure of whether a content refresh caused improvement. Only 3,611,061 of 9,841,378 March rows have GSC data marked available, and only 413,966 have GA4 data marked available. GA4 coverage is therefore much lower than GSC coverage. Missing GA4 measurements must not automatically be interpreted as zero activity. A future-outcome label will need a defined time window and a strict separation between features available at the decision moment and later outcomes.

In [20]:
# QUERY 1: Verify the row grain
# One row should represent report_date × client × content.
grain_query = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            report_date,
            client_hash_id,
            content_hash_id
        )) AS distinct_grain_keys
    FROM march
""").df()

print("QUERY 1 — GRAIN VERIFICATION")
display(grain_query)

# QUERY 2: Verify the March slice row count and date span
window_query = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM march
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
""").df()

print("QUERY 2 — ROW COUNT AND DATE SPAN")
display(window_query)

# QUERY 3: Verify data availability using IS TRUE
availability_query = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM march
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
""").df()

print("QUERY 3 — AVAILABILITY")
display(availability_query)

# Brief verification summary
total = int(grain_query.loc[0, "total_rows"])
distinct_keys = int(grain_query.loc[0, "distinct_grain_keys"])

print(
    "Grain unique:",
    total == distinct_keys
)
print(
    "March date span:",
    window_query.loc[0, "first_date"],
    "to",
    window_query.loc[0, "last_date"]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — GRAIN VERIFICATION


,total_rows,distinct_grain_keys
0,9841378,9841378


QUERY 2 — ROW COUNT AND DATE SPAN


,row_count,first_date,last_date,distinct_dates
0,9841378,2026-03-01,2026-03-31,31


QUERY 3 — AVAILABILITY


,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


Grain unique: True
March date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


In [21]:
# Inspect potential additional GSC features
gsc_candidates = con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(*) FILTER (
            WHERE gsc_sum_position IS NULL
        ) AS missing_sum_position,
        COUNT(*) FILTER (
            WHERE client_has_gsc IS NULL
        ) AS missing_client_has_gsc,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NULL
        ) AS missing_availability,
        COUNT(*) FILTER (
            WHERE gsc_impressions = 0
        ) AS zero_impressions,
        COUNT(*) FILTER (
            WHERE gsc_clicks = 0
        ) AS zero_clicks
    FROM march
    WHERE gsc_data_available IS TRUE
""").df()

display(gsc_candidates)

,rows,missing_sum_position,missing_client_has_gsc,missing_availability,zero_impressions,zero_clicks
0,3611061,0,0,0,0,3193080


In [22]:
# List the available monthly performance partitions
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
)

monthly_files = [
    path for path in files
    if path.startswith("fact_content_daily_performance/month=")
    and path.endswith(".parquet")
]

for path in sorted(monthly_files):
    print(path)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

## 4. Data limits
**Data limits:** The March 2026 partition contains daily page-level observations, and GSC availability is much lower than the total row count. Among GSC-available rows, 3,193,080 have zero clicks, so click-based outcomes may be sparse. GA4 coverage is substantially lower than GSC coverage, so GA4-based conclusions may apply to a narrower subset. These data are observational and do not prove that refreshing a page causes traffic to improve.

In [23]:
# Quantify the availability limitation in the March 2026 slice

availability_limits = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available
    FROM march
""").df()

display(availability_limits)

gsc_rate = availability_limits.loc[0, "gsc_available"] / availability_limits.loc[0, "total_rows"]
ga4_rate = availability_limits.loc[0, "ga4_available"] / availability_limits.loc[0, "total_rows"]

print(f"GSC availability: {gsc_rate:.1%}")
print(f"GA4 availability: {ga4_rate:.1%}")


,total_rows,gsc_available,ga4_available,both_available
0,9841378,3611061,413966,364347


GSC availability: 36.7%
GA4 availability: 4.2%


In [24]:
# FINAL CHECK: five features + deliberate leakage demonstration

import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

if "monthly" not in globals():
    raise ValueError(
        "Run the March/April outcome cell first. "
        "It creates the monthly DataFrame."
    )

# Keep rows with a defined future-outcome proxy
data = monthly.dropna(subset=["future_decline_proxy"]).copy()
data["future_decline_proxy"] = data["future_decline_proxy"].astype(int)

# Five candidate features available from March data only
features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_days",
    "march_available_days",
]

X = data[features].fillna(0)
y = data["future_decline_proxy"]

print("Rows:", f"{len(data):,}")
print("Future-decline proxy rate:", f"{y.mean():.1%}")
print("\nFive features:")
for feature in features:
    print(f"- {feature}: derived from March observations available by the decision date.")

# Demonstrate leakage by including the target itself as an invalid feature.
# This is intentionally dishonest and is used ONLY to show the trap.
X_leaky = X.copy()
X_leaky["LEAKED_LABEL_DO_NOT_USE"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

leaky_model = DecisionTreeClassifier(max_depth=2, random_state=42)
leaky_model.fit(X_train, y_train)
leaky_score = accuracy_score(y_test, leaky_model.predict(X_test))

print("\nDeliberate leakage experiment")
print("Score with target-derived column:", f"{leaky_score:.3f}")

# Remove the leaked column; keep only honest March features.
X_honest = data[features].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)

honest_model = DecisionTreeClassifier(max_depth=2, random_state=42)
honest_model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_model.predict(X_test))

print("Honest score after removing leaked column:", f"{honest_score:.3f}")
print("Final feature frame contains only March features:", list(X_honest.columns))

Rows: 158,549
Future-decline proxy rate: 47.8%

Five features:
- march_impressions: derived from March observations available by the decision date.
- march_clicks: derived from March observations available by the decision date.
- march_avg_position: derived from March observations available by the decision date.
- march_days: derived from March observations available by the decision date.
- march_available_days: derived from March observations available by the decision date.

Deliberate leakage experiment
Score with target-derived column: 1.000
Honest score after removing leaked column: 0.602
Final feature frame contains only March features: ['march_impressions', 'march_clicks', 'march_avg_position', 'march_days', 'march_available_days']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.